# Module 4 Homework — Stock Markets Analytics Zoomcamp 2026

Lesson reference: [`04-trading-strategy-and-simulation/[2026]_Module_04_Colab_Trading_Simulations.ipynb`](https://github.com/DataTalksClub/stock-markets-analytics-zoomcamp/blob/main/04-trading-strategy-and-simulation/%5B2026%5D_Module_04_Colab_Trading_Simulations.ipynb)

Homework spec: [`cohorts/2026/homework4.md`](https://github.com/DataTalksClub/stock-markets-analytics-zoomcamp/blob/main/cohorts/2026/homework4.md)
Submission form: https://courses.datatalks.club/sma-zoomcamp-2026/homework/hw04

## Pipeline (shared)

Reproduces the lesson end to end: features, dummies, 70/15/15 temporal split, hand rules `pred0`–`pred4`, the Decision Tree / Random Forest predictors `pred5`–`pred11`, and the $100-per-signal simulation. Logistic Regression and the neural network are skipped since they don't feed the simulation. Hand rules and the depth-4/7 trees match the lesson's Colab outputs exactly; the deeper models (depth-10 tree, Random Forest) differ slightly because tie-breaking between equal splits varies across platforms and sklearn versions.

In [1]:
"""
Shared pipeline for Module 4: reproduces the lesson notebook end to end
(features, dummies, temporal split, hand rules, Decision Tree / Random Forest
predictors, and the $100-per-signal trading simulation).

Knobs for the homework:
  - categorical: which columns get dummies (Q1 drops Ticker and month_wom)
  - target_threshold: growth_future_30d cut-off for the target (Q3 uses 1.022)
  - rf_thresholds: extra Random Forest decision rules (Q2)

The course parquet is shared with Module 3 (cached in 03-modeling/data/).
"""
import importlib.util
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

# reuse Module 3's loader and feature lists (its file is also named prep.py)
_spec = importlib.util.spec_from_file_location("m3_prep", Path.cwd().parent / "03-modeling" / "prep.py")
m3 = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(m3)
MACRO, TECHNICAL_INDICATORS, load_df_full, temporal_split = m3.MACRO, m3.TECHNICAL_INDICATORS, m3.load_df_full, m3.temporal_split

TARGET = "is_positive_growth_30d_future"
CATEGORICAL = ["Month", "Weekday", "Ticker", "ticker_type", "month_wom"]
CUSTOM_NUMERICAL = ["SMA10", "SMA20", "growing_moving_average", "high_minus_low_relative",
                    "volatility", "ln_volume", "div_payout", "stock_split"]


def numerical_features(df):
    growth = [k for k in df.keys() if k.startswith("growth_") and "future" not in k]
    patterns = [k for k in df.keys() if "cdl" in k]
    return growth + TECHNICAL_INDICATORS + patterns + CUSTOM_NUMERICAL + MACRO


def build_new_df(categorical=CATEGORICAL, target_threshold=1.0):
    """Returns (new_df, features) like the lesson notebook."""
    df_full = load_df_full()
    df_full["ln_volume"] = np.log(df_full.Volume + 1e-6)
    df_full["div_payout"] = (df_full.Dividends > 0).astype(int)
    df_full["stock_split"] = (df_full["Stock Splits"] > 0).astype(int)
    df_full[TARGET] = (df_full.growth_future_30d > target_threshold).astype(int)

    df = df_full[df_full.Date >= "2000-01-01"].copy()
    df["Month"] = df.Month.dt.month_name()
    df["Weekday"] = df.Weekday.astype(str)
    df["month_wom"] = df.Month + "_w" + ((df.Date.dt.day - 1) // 7 + 1).astype(str)

    dummies = pd.get_dummies(df[categorical], dtype="int32")
    new_df = pd.concat([df, dummies], axis=1)
    new_df["split"] = temporal_split(new_df)

    new_df["pred0_manual_cci"] = (new_df.cci > 200).astype(int)
    new_df["pred1_manual_prev_g1"] = (new_df.growth_30d > 1).astype(int)
    new_df["pred2_manual_prev_g1_and_snp"] = ((new_df.growth_30d > 1) & (new_df.growth_snp500_30d > 1)).astype(int)
    new_df["pred3_manual_dgs10_5"] = ((new_df.DGS10 <= 4.5) & (new_df.DGS5 <= 4)).astype(int)
    new_df["pred4_manual_dgs10_fedfunds"] = ((new_df.DGS10 > 4) & (new_df.FEDFUNDS <= 4.795)).astype(int)
    return new_df.copy(), numerical_features(df_full) + dummies.columns.to_list()


def clean_X(df, features):
    return df[features].replace([np.inf, -np.inf], np.nan).fillna(0)


def add_model_predictions(new_df, features, rf_thresholds=()):
    """Fits the lesson's models on train+validation and adds pred5..pred11 (+ extra RF rules).

    Returns (rf_best, clf_best) for further analysis.
    """
    train_valid = new_df.split.isin(["train", "validation"])
    X_train_valid, y_train_valid = clean_X(new_df[train_valid], features), new_df.loc[train_valid, TARGET]
    X_all = clean_X(new_df, features)

    def tree(depth):
        return DecisionTreeClassifier(max_depth=depth, random_state=42).fit(X_train_valid, y_train_valid)

    new_df["pred5_clf_10"] = tree(10).predict(X_all)
    new_df["pred6_clf_4"] = tree(4).predict(X_all)
    clf_best = tree(7)
    new_df["pred7_clf_second_best"] = clf_best.predict(X_all)

    proba = clf_best.predict_proba(X_all)[:, 1]
    new_df["pred8_clf_second_best_rule_56"] = (proba >= 0.56).astype(int)
    new_df["pred9_clf_second_best_rule_66"] = (proba >= 0.66).astype(int)

    rf_best = RandomForestClassifier(n_estimators=100, max_depth=13, random_state=42, n_jobs=-1)
    rf_best.fit(X_train_valid, y_train_valid)
    proba = rf_best.predict_proba(X_all)[:, 1]
    new_df["pred10_rf_best_rule_55"] = (proba >= 0.55).astype(int)
    new_df["pred11_rf_best_rule_65"] = (proba >= 0.65).astype(int)
    for i, t in enumerate(rf_thresholds, start=12):
        new_df[f"pred{i}_rf_best_rule_{round(t * 100)}"] = (proba >= t).astype(int)
    return rf_best, clf_best


def simulate(new_df):
    """Lesson simulation: $100 per positive prediction on TEST, held 30 days, 0.2% fees."""
    test = new_df[new_df.split == "test"]
    rows = []
    for pred in [k for k in new_df.keys() if k.startswith("pred")]:
        trades = test[test[pred] == 1]
        gross = (100 * (trades.growth_future_30d - 1)).sum()
        fees = -0.2 * len(trades)
        net = (100 * (trades.growth_future_30d - 1) - 0.2).sum()  # like the notebook: skips NaN future growth
        daily = trades.groupby("Date")[pred].count()
        capital = 100 * 30 * daily.quantile(0.75)
        rows.append({
            "prediction": pred,
            "count_investments": len(trades),
            "gross_rev": gross,
            "fees": fees,
            "net_rev": net,
            "avg_investments_per_day": daily.mean(),
            "capital": capital,
            "CAGR": ((capital + net) / capital) ** (1 / 4) if capital else np.nan,
        })
    return pd.DataFrame(rows).set_index("prediction")


## Question 1: Less Features is More

**Main task:** CAGR for `pred10_rf_best_rule_55` with `Ticker` and `month_wom` removed from the dummies (301 → 208 features).

The spec's 1.1195 is the original best ML predictor (`pred5_clf_10` = 1.1196 in the lesson's Colab run), not the original `pred10` (1.0936). Locally `pred10` reaches 1.1189 — closest form option: **1.121**.

In [2]:
"""
Question 1: Less Features is More

Find the CAGR for pred10_rf_best_rule_55 using the new setup
(Ticker and month_wom removed from CATEGORICAL before generating dummies).
"""
import pandas as pd


pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)


def run(categorical):
    new_df, features = build_new_df(categorical)
    print(f"Features: {len(features)}")
    add_model_predictions(new_df, features)
    return simulate(new_df)


def main():
    short = [c for c in CATEGORICAL if c not in ("Ticker", "month_wom")]
    results = {name: run(cats) for name, cats in [("original", CATEGORICAL), ("short", short)]}

    print(pd.concat(results, axis=1)[[(n, c) for c in ("CAGR", "avg_investments_per_day") for n in results]].round(4))
    # spec's 1.1195 is the original best ML predictor (pred5_clf_10 in the notebook's Colab run)
    best_original = results["original"].CAGR.idxmax()
    cagr = results["short"].loc["pred10_rf_best_rule_55", "CAGR"]
    print(f"\nOriginal best: {best_original} = {results['original'].CAGR.max():.4f} (Colab: pred5_clf_10 = 1.1196)")
    print(f"ANSWER Q1: pred10_rf_best_rule_55 CAGR = {cagr:.4f}")


if __name__ == "__main__":
    main()


Features: 301


Features: 208


                              original   short                original                   short
                                  CAGR    CAGR avg_investments_per_day avg_investments_per_day
prediction                                                                                    
pred0_manual_cci                1.0628  1.0628                  1.8758                  1.8758
pred1_manual_prev_g1            1.1191  1.1191                 18.4212                 18.4212
pred2_manual_prev_g1_and_snp    1.0900  1.0900                 21.7151                 21.7151
pred3_manual_dgs10_5            1.0837  1.0837                 31.9478                 31.9478
pred4_manual_dgs10_fedfunds     1.0371  1.0371                 31.9058                 31.9058
pred5_clf_10                    1.1185  1.1270                  9.0317                 10.3516
pred6_clf_4                     1.0494  1.0494                 30.3500                 30.3500
pred7_clf_second_best           1.0668  1.0539    

## Question 2: Most Efficient Random Forest Decision Rule

**Main task:** threshold of the best-CAGR Random Forest rule with < 5 average trades per day.

Locally the RF trades more than in Colab (6.4/day at 0.55 vs. the spec's "slightly more than 5"), so the first rule under 5 is 0.59 here. Each threshold step removes ~0.3–0.5 trades/day, so in Colab 0.56 is the first rule under 5 and keeps the best CAGR — answer **0.56**.

In [3]:
"""
Question 2: Most Efficient Random Forest Decision Rule

On the Q1 setup (no Ticker / month_wom dummies), add RF rules for thresholds
0.56..0.64. Which threshold gives the best CAGR with < 5 average trades per day?
"""
import numpy as np
import pandas as pd


pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)


def main():
    short = [c for c in CATEGORICAL if c not in ("Ticker", "month_wom")]
    new_df, features = build_new_df(short)
    add_model_predictions(new_df, features, rf_thresholds=np.arange(56, 65) / 100)

    results = simulate(new_df).filter(like="_rf_", axis=0)
    test = new_df[new_df.split == "test"]
    results["precision"] = [test.loc[test[p] == 1, TARGET].mean() for p in results.index]
    results = results.sort_values("CAGR", ascending=False)
    print(results[["count_investments", "precision", "avg_investments_per_day", "CAGR"]].round(4))

    best = results[results.avg_investments_per_day < 5].index[0]
    print(f"\nANSWER Q2: {best} -> threshold 0.{best.split('_')[-1]}")


if __name__ == "__main__":
    main()


                        count_investments  precision  avg_investments_per_day    CAGR
prediction                                                                           
pred10_rf_best_rule_55               3776     0.6210                   6.4000  1.1189
pred12_rf_best_rule_56               2847     0.6171                   5.8460  1.1009
pred13_rf_best_rule_57               2070     0.6256                   5.3906  1.0833
pred14_rf_best_rule_58               1514     0.6268                   5.1322  1.0725
pred15_rf_best_rule_59               1101     0.6322                   4.9152  1.0543
pred17_rf_best_rule_61                568     0.6373                   3.9172  1.0432
pred16_rf_best_rule_60                786     0.6234                   4.4659  1.0376
pred18_rf_best_rule_62                372     0.6371                   3.4128  1.0352
pred20_rf_best_rule_64                154     0.6429                   2.4062  1.0246
pred19_rf_best_rule_63                241     0.6266  

## Question 3: Redefining the Growth Target

**Main task:** number of trades for `pred5_clf_10` with target `growth_future_30d > 1.022`.

Form options are 6–9, so "number of trades" is average trades per day: **8**.

In [4]:
"""
Question 3: Redefining the Growth Target

With is_positive_growth_30d_future = growth_future_30d > 1.022 (original
dummies), what is the number of trades for the best predictor pred5_clf_10?
"""
import pandas as pd


pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)


def main():
    new_df, features = build_new_df(target_threshold=1.022)
    print(f"Class 1 share: {new_df[TARGET].mean():.3f} (spec ~0.50)")
    add_model_predictions(new_df, features)

    results = simulate(new_df).sort_values("CAGR", ascending=False)
    print(results[["count_investments", "avg_investments_per_day", "CAGR"]].round(4))

    pred5 = results.loc["pred5_clf_10"]
    print(f"\nANSWER Q3: pred5_clf_10 avg trades per day = {pred5.avg_investments_per_day:.2f} "
          f"(rounded {round(pred5.avg_investments_per_day)}), total {int(pred5.count_investments)}")


if __name__ == "__main__":
    main()


Class 1 share: 0.491 (spec ~0.50)


                               count_investments  avg_investments_per_day    CAGR
prediction                                                                       
pred5_clf_10                                7201                   7.8528  1.1200
pred1_manual_prev_g1                       19158                  18.4212  1.1191
pred2_manual_prev_g1_and_snp               15244                  21.7151  1.0900
pred7_clf_second_best                      10532                  12.6739  1.0866
pred3_manual_dgs10_5                       15910                  31.9478  1.0837
pred8_clf_second_best_rule_56              10276                  12.4860  1.0822
pred9_clf_second_best_rule_66               9746                  13.4799  1.0687
pred0_manual_cci                             921                   1.8758  1.0628
pred6_clf_4                                 7209                  30.8077  1.0527
pred10_rf_best_rule_55                       143                   1.7024  1.0426
pred4_manual_dgs

## Question 4: Describe Your Ideal Trading Strategy (Exploratory)

In [5]:
"""
Question 4: [Exploratory] Describe Your Ideal Trading Strategy
"""

ANSWER = """
Each day, invest only in the top 3 stocks by Random Forest probability (>= 0.60),
and only if the decision tree also predicts growth. Split capital across the 30-day
holding window, weight by confidence, and cap each position at 30%. Use a 10%
stop-loss and skip new entries when the S&P 500's 30-day trend is negative.
"""


if __name__ == "__main__":
    print(ANSWER)



Each day, invest only in the top 3 stocks by Random Forest probability (>= 0.60),
and only if the decision tree also predicts growth. Split capital across the 30-day
holding window, weight by confidence, and cap each position at 30%. Use a 10%
stop-loss and skip new entries when the S&P 500's 30-day trend is negative.

